Імпорт бібліотек


In [1]:
%pip install -U langchain langgraph langchain-openai

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
%pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import os
import uuid

from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent

from langgraph.store.memory import InMemoryStore
from langgraph.config import get_store
from dotenv import load_dotenv

In [4]:
from pathlib import Path

env_path = Path(".env")
if not env_path.exists():
    env_path = Path("HW_6_AI_agent") / ".env"

load_dotenv(env_path)

api_key = os.getenv("OPENAI_API_KEY")

if api_key:
    print("API key loaded successfully")
else:
    print("API key not found")

API key loaded successfully


створення моделі

In [5]:
from langchain_openai import ChatOpenAI

if api_key.startswith("sk-or-"):
    model = ChatOpenAI(
        model="openai/gpt-4o-mini",
        api_key=api_key,
        base_url="https://openrouter.ai/api/v1",
        temperature=0,
    )
else:
    model = ChatOpenAI(
        model="gpt-4o-mini",
        api_key=api_key,
        temperature=0,
    )

print("Model created successfully")

Model created successfully


In [6]:
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()

print("Store created successfully")

Store created successfully


Створюєм інструменти

In [7]:
from langchain_core.tools import tool
from langgraph.config import get_store
import uuid

@tool
def add_task(title: str) -> str:
    """Додає нове завдання до TODO-списку.

    Args:
        title: Назва завдання, яке потрібно додати.
    """
    store = get_store()

    task_id = str(uuid.uuid4())

    task_data = {
        "title": title
    }

    store.put(
        ("tasks",),
        task_id,
        task_data
    )

    return f'Завдання "{title}" було успішно додано. ID: {task_id}'

In [8]:
@tool
def list_tasks() -> str:
    """Показує всі збережені завдання з TODO-списку."""
    store = get_store()

    tasks = store.search(("tasks",))

    if not tasks:
        return "Список завдань порожній."

    result = "Ось всі ваші завдання:\n\n"

    for i, task in enumerate(tasks, 1):
        task_id = task.key
        title = task.value["title"]

        result += f"{i}. {title} [ID: {task_id}]\n"

    return result

In [9]:
@tool
def delete_task(query: str) -> str:
    """Видаляє завдання за повним ID, коротким ID або частиною назви.

    Args:
        query: ID завдання, короткий ID або текст з назви завдання.
    """
    store = get_store()

    tasks = store.search(("tasks",))
    query_normalized = query.strip().lower()
    ignored_words = {"видали", "видалити", "завдання", "задачу", "задача", "про", "будь", "ласка"}
    query_words = [
        word
        for word in query_normalized.replace(":", " ").replace(",", " ").split()
        if len(word) > 2 and word not in ignored_words
    ]

    matches = []
    for task in tasks:
        task_id = task.key
        title = task.value["title"]
        title_normalized = title.lower()

        if (
            task_id == query_normalized
            or task_id.startswith(query_normalized)
            or query_normalized in title_normalized
            or any(word in title_normalized for word in query_words)
        ):
            matches.append(task)

    if not matches:
        return "Завдання не знайдено."

    unique_matches = {task.key: task for task in matches}
    matches = list(unique_matches.values())

    if len(matches) > 1:
        titles = ", ".join(task.value["title"] for task in matches)
        return f"Знайдено кілька завдань: {titles}. Уточніть, яке видалити."

    task = matches[0]
    task_title = task.value["title"]
    store.delete(("tasks",), task.key)

    return f'Завдання "{task_title}" було видалено.'

In [10]:
tools = [
    add_task,
    list_tasks,
    delete_task
]

agent = create_agent(
    model=model,
    tools=tools,
    store=store,
    system_prompt="""
Ти TODO-агент, який керує списком справ українською мовою.

Твої правила:
1. Якщо користувач просить додати завдання, використовуй add_task.
2. Якщо користувач просить показати список або питає, що залишилось, використовуй list_tasks.
3. Якщо користувач просить видалити завдання за ID, коротким ID або змістом, використовуй delete_task.
4. Для фраз на кшталт "видали завдання про хліб" передай у delete_task ключову частину назви, наприклад "хліб".
5. Після видалення можеш коротко підтвердити, яке завдання видалено.
6. Відповідай коротко й українською.
"""
)

print("Agent created successfully")

Agent created successfully


Тест


In [11]:
tests = [
    "Додай: купити хліб",
    "Додай: подзвонити лікарю",
    "Покажи всі завдання",
    "Видали завдання про хліб",
    "Що залишилось?"
]

for i, test in enumerate(tests, 1):
    print(f"\n{'='*50}")
    print(f"TEST {i}: {test}")
    print('='*50)

    result = agent.invoke({
        "messages": [
            {"role": "user", "content": test}
        ]
    })

    print(result["messages"][-1].content)


TEST 1: Додай: купити хліб
Завдання "купити хліб" було успішно додано.

TEST 2: Додай: подзвонити лікарю
Завдання "подзвонити лікарю" було успішно додано.

TEST 3: Покажи всі завдання
Ось всі ваші завдання:

1. купити хліб
2. подзвонити лікарю

TEST 4: Видали завдання про хліб
Завдання "купити хліб" було видалено.

TEST 5: Що залишилось?
Ось ваше завдання: 

1. подзвонити лікарю.
